# Antarctic AR Identifier/Catalog Building
#### James Butler and Michelle Maclennan, June 2024

### Motivation

Jonathan Wille's AR catalogs identify points in space-time associated with AR conditions (using MERRA-2 data), but these points have-yet to be clustered into different storms to facilitate event-based analyses. This notebook explores one procedure to partition these AR points into storms. 

### Setup

In [65]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import dask
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances
import math

import pygmt

import matplotlib.path as mpath
import matplotlib.colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

from matplotlib import animation
from IPython.display import Video
from matplotlib.cm import prism

from ipywidgets import IntProgress
from IPython.display import display

In [66]:
# Load up the AR catalogs
curwd = os.getcwd()
catalog_paths = str(Path(curwd).parents[0]) + '/data/ar_catalogs/*.nc'
full_catalog = xr.open_mfdataset(catalog_paths)

# Load up the AIS mask
mask_path = str(Path(curwd).parents[0]) + '/data/antarctic_masks/AIS_Full_basins_Zwally_MERRA2grid_new.nc'
full_ais_mask = xr.open_dataset(mask_path).Zwallybasins > 0

### Preprocessing

In [67]:
# get rid of all non-antarctic points
# making the same slice decisions as Michelle
catalog = full_catalog.sel(lat=slice(-86, -39))
ais_mask = full_ais_mask.sel(lat=slice(-86, -39))

In [68]:
# subset for a year just to showcase the algorithm
start_date = '1980-01-01T03:00:00.000000000'
end_date = '1980-12-31T03:00:00.000000000'
practice_catalog = catalog.sel(time=slice(start_date, end_date))
# get rid of all time steps for which there is no AR present
is_ar_time = practice_catalog.ar_binary_tag.any(dim = ['lat', 'lon'])
practice_catalog = practice_catalog.sel(time=is_ar_time).ar_binary_tag

### Clustering

Strategy consists of two stages, one which partitions AR points at each individual time step into different storms (*Spatial Clustering using DBSCAN*), and one which stitches the clusters in each time-specific partition across time (*Spatiotemporal Clustering using ST-DBSCAN*). 

#### Aside: Useful functions for clustering

In [69]:
# homegrown arctan function to make sure that, for a given x and y, the
# the angle corresponds to the correct hemisphere of the unit circle
def arctan(x, y):
    if y/x > 0:
        if x > 0:
            return(np.arctan(y/x))
        else:
            return(np.arctan(y/x)-np.pi)
    else:
        if x > 0:
            return(np.arctan(y/x))
        else:
            return(np.pi+np.arctan(y/x))
    

# following wikipedia article on circular mean
# standard arithmetic means of non-euclidean (i.e. cyclic) spaces can behave badly
# instead, convert angles to unit vectors in R3, average components, find angles of avg. vector
def average_angle(subdf):
    lats = np.radians(subdf.lats)
    lons = np.radians(subdf.lons)

    x = np.cos(lats)*np.cos(lons)
    y = np.cos(lats)*np.sin(lons)
    z = np.sin(lats)
    
    avg_x = np.mean(x)
    avg_y = np.mean(y)
    avg_z = np.mean(z)

    avg_lat = np.arcsin(avg_z)
    avg_lon = arctan(avg_x, avg_y)

    return (subdf.name, np.degrees(avg_lat), np.degrees(avg_lon))

#### Stage 1: Spatial Clustering using DBSCAN

First, go through each AR timestep and partition the AR points present into cluster(s). For each set of clusters generated for each time step, save the **(1) lat/lon pairs associated with the clusters**, **(2) the mean lat/lon of the cluster (cluster centroid of sorts)**, and **(3) a random sample of points to act as a "representative" sample of lat/lon pairs for that cluster** (will greatly facilitiate spatiotemporal clustering stage).

In [70]:
# times to loop through to cluster
times = practice_catalog.time.to_numpy()
# instantiate empty list with each index corresponding to a time step
# each entry will consist of a dataframe of clusters subdivided at that particular time
cluster_infos = [None]*len(times)

# instantiate progress bar for for-loop
max_count = len(times)
f = IntProgress(min=0, max=max_count)

display(f)

# from each cluster created, number of representative points to randomly sample
n_rep_pts = 10

# for each AR time
for i in range(len(times)):
    
    time_slice = practice_catalog.sel(time = times[i])
    # find lats/lons of AR points in this time step
    inds = np.argwhere(time_slice.to_numpy() == 1)
    storm_lats = time_slice.lat[inds[:,0]]
    storm_lons = time_slice.lon[inds[:,1]]

    # cluster spatially using DBSCAN, synoptic scale neighborhood size
    synoptic_scale = (10**3)/2
    km_per_radian = 6.371*(10**3) # arclength (km) on earth subtended by 1 radian  
    eps = synoptic_scale/km_per_radian # converted to radians for Haversine metric
    clustering = DBSCAN(eps=eps, min_samples=5, metric='haversine').fit_predict(X=np.column_stack((np.radians(storm_lats), np.radians(storm_lons))))
    fixed_time_df = pd.DataFrame({'lats': storm_lats, 'lons': storm_lons, 'cluster': clustering})

    # get the average lat-lon of each cluster, according to average_angle function above
    avg_positions = pd.DataFrame(fixed_time_df.groupby('cluster').apply(average_angle, include_groups=False).to_list(), 
                             columns=['cluster', 'mean_lat', 'mean_lon'])

    # randomly sample n_rep_pts-many points (without replacement) from each cluster and store
    rep_pts = fixed_time_df.groupby('cluster', as_index=False)[['lats', 'lons']].agg(lambda x: list(np.random.choice(x, min(n_rep_pts, len(x)), replace=False)))
    rep_pts.rename(columns={'lats':'rep_lats', 'lons':'rep_lons'}, inplace=True)

    rep_pts_df = pd.merge(rep_pts, avg_positions, on='cluster')

    # aggregate ALL lats and lons for each cluster into lists as well
    cluster_info = fixed_time_df.groupby('cluster', as_index=False)[['lats', 'lons']].agg(list)
    
    
    # combine all this info into single dataframe
    cluster_info = pd.merge(cluster_info, rep_pts_df, on='cluster')
    # add time into column
    cluster_info['time'] = times[i]
    # add this time-specific df into list of dfs
    cluster_infos[i] = cluster_info

    f.value += 1

IntProgress(value=0, max=888)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f832e43c610>>
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# stitch list of dataframes across time into big dataframe
cluster_infos_df = pd.concat(cluster_infos, axis=0)
cluster_infos_df.reset_index(drop=True, inplace=True)
# preallocate empty column for cluster labels for each AR timestep
ar_pt_df = cluster_infos_df[['mean_lat', 'mean_lon', 'rep_lats', 'rep_lons', 'time']]
ar_pt_df['cluster'] = np.full(cluster_infos_df.shape[0], np.nan)

### Spatiotemporal Clustering of Mean Locations, using ST-DBSCAN

Great! Now we have partitions of AR points for each time step, but we still need to establish correspondence between elements of the partitions *across* time steps. To do this, we use the ST-DBSCAN algorithm as showcased in ST-DBSCAN: *An algorithm for clustering spatio-temporal data* by Birant and Kut (2007). This algorithm searches for clusters based on density of points as in the spatial clustering step, but also does so across time. Specifically, we use ST-DBSCAN to cluster the set of "representative" points sampled from each cluster created in the last step.

**Why don't we just use ST-DBSCAN on the full dataset of AR points, so that we can avoid the spatial clustering step altogether?**

There is no package implementation of ST-DBSCAN, so I had to hard-code up the algorithm described in Birant and Kut (2007). Unfortunately, this is a naive hard-coding that doesn't use any fancy approximation methods to speed up the algorithm, and so it runs VERY slow on large datasets. Instead, we run this home-grown code directly on the dataset consisting of the representative points of the clusters determined in Stage 1, which helps save on time since we are dealing with a MUCH smaller dataset.


##### Aside: useful helper function for ST-DBSCAN algorithm

In [ ]:
def retrieve_neighbors(object, data, eps_space, eps_time):
    '''
    object: a single row of dataframe with time, mean_lat, mean_lon columns;
        represents one of possibly many ARs at a single time step
    data: the rest of the dataset to cluster
    eps_space: neighborhood size in space (angular size)
    eps_time: neighborohod size in time (in days)
    '''
    
    obj_time = object['time'].iloc[0]
    obj_loc = object[['lat', 'lon']]

    # find neighbors in time
    time_neighbors = data.loc[np.abs((data['time'] - obj_time).dt.total_seconds()/86400) <= eps_time]
    # among time neighbors, find space neighbors
    st_neighbors = time_neighbors.loc[haversine_distances(time_neighbors[['lat', 'lon']], obj_loc) <= eps_space]

    return(st_neighbors)

##### Preprocess for ST-DBSCAN algorithm
Need to unpack the data frame previously made into one big dataframe where each row consists of a randomly sampled AR point. Data must be in this format to use ST-DBSCAN

In [ ]:
# loop through the dataframe made and unpack all of the representative points sampled for each cluster
# resulting dataframe will have rows consisting of representative storm point
unpacked_indices = []
unpacked_lats = []
unpacked_lons = []
unpacked_times = []

for index in list(ar_pt_df.index):
    num_pts = len(ar_pt_df.loc[index].rep_lats) + 1
    unpacked_indices = unpacked_indices + [index]*num_pts
    unpacked_times = unpacked_times + [ar_pt_df.loc[index].time]*num_pts
    unpacked_lats = unpacked_lats + list(np.radians(ar_pt_df.loc[index].rep_lats)) + [np.radians(ar_pt_df.loc[index].mean_lat)]
    unpacked_lons = unpacked_lons + list(np.radians(ar_pt_df.loc[index].rep_lons)) + [np.radians(ar_pt_df.loc[index].mean_lon)]

unpacked_df = pd.DataFrame({'cluster':np.full(len(unpacked_indices), np.nan), 'space_cluster':unpacked_indices, 'time':unpacked_times, 'lat':unpacked_lats, 'lon':unpacked_lons})

##### Running through the ST-DBSCAN

In [ ]:
# hyperparams to ST-DBSCAN algorithm
min_pts = 5
noise_label = 7777777
cluster_label = 0

# same spatial scale as the spatial clustering
synoptic_scale = (10**3)/2
km_per_radian = 6.371*(10**3) # arclength (km) on earth subtended by 1 radian  
eps_space = synoptic_scale/km_per_radian

# neighboring points only considered within 18 hours of current point
eps_time = 18/24

ar_pt_df = unpacked_df

# for each point
for i in range(ar_pt_df.shape[0]):

    cur_obj = ar_pt_df.iloc[[i]]
    # if either unclustered or noise
    if math.isnan(ar_pt_df.loc[i, 'cluster']) or ar_pt_df.loc[i, 'cluster'] == noise_label:
        
        neighbors = retrieve_neighbors(cur_obj, ar_pt_df, eps_space, eps_time)
        # if less than min_pts neighbors, accounting for the point itself
        if neighbors.shape[0] < min_pts + 1:
            ar_pt_df.loc[i, 'cluster'] = noise_label
        # otherwise, start new cluster and label your neighbors accordingly
        else:
            cluster_label = cluster_label + 1
            ar_pt_df.loc[neighbors.index, 'cluster'] = cluster_label
            # indices to keep track of which points will be in cluster
            cluster_inds = list(neighbors.drop(i).index)

            # while we still have unprocessed cluster points
            while cluster_inds:
                new_cur_obj = ar_pt_df.loc[[cluster_inds.pop()]]
                new_neighbors = retrieve_neighbors(new_cur_obj, ar_pt_df, eps_space, eps_time)
                
                if new_neighbors.shape[0] >= min_pts + 1:
                    # if neighboring point unlabelled, endow with cluster label
                    unlabelled = new_neighbors.loc[new_neighbors['cluster'].isnull()]
                    ar_pt_df.loc[unlabelled.index, 'cluster'] = cluster_label
                    # add these newly clustered points to list of unprocessed cluster points
                    cluster_inds = cluster_inds + list(unlabelled.index)

In [ ]:
cluster_assignments = ar_pt_df.groupby('space_cluster')['cluster'].apply(lambda series: series.value_counts().idxmax())

In [ ]:
# add cluster membership column back to original df
cluster_infos_df['cluster'] = cluster_assignments

## Postprocessing 

After clustering spatiotemporally, do things like filter out noise clusters, identify landfalling time steps (over the AIS), etc.

In [ ]:
# remove noise clusters
# noise cluster meaning they are one-off ARs
cluster_infos_df = cluster_infos_df[cluster_infos_df['cluster'] != noise_label]

In [ ]:
# get ais points
ais_mask_lats = ais_mask.lat[np.where(ais_mask.to_numpy())[0]].to_numpy()
ais_mask_lons = ais_mask.lon[np.where(ais_mask.to_numpy())[1]].to_numpy()
ais_pts = set(zip(ais_mask_lats, ais_mask_lons))

# determine which steps are landfalling and which are not, given a row of dataframe
def is_landfalling(row):
    lats = np.array(np.degrees(row.lats))
    lons = np.array(np.degrees(row.lons))

    storm_pts = set(zip(lats, lons))

    return(bool(storm_pts & ais_pts))

# add is_landfalling column to each AR at each time step
cluster_infos_df['is_landfalling'] = cluster_infos_df.apply(is_landfalling, axis=1)

In [ ]:
cluster_infos_df.to_pickle('cluster_infos_1980.pkl')

### Diagnostic Movie

Make a movie of all our identified ARs to see if our clustering procedure was effective! Also, overlay countours corresponding to Jonathan's raw catalogs to ensure we are picking up everything. Use the 1980 data as my "training" data to perfect the algorithm. Will test out on other year(s) of the dataset.

In [ ]:
plt_df = cluster_infos_df

unique_clusters = plt_df['cluster'].unique()
color_mapping = {unique_clusters[j]:prism(j/len(unique_clusters)) for j in range(len(unique_clusters)) }

times = np.array(pd.date_range('1980-01-02T03:00:00.000000000', '1980-12-31T03:00:00.000000000', freq='3h'))



def plt_time(j):
    time_pt = times[j]

    if (time_pt == plt_df.time).any():
        dat = plt_df[plt_df['time'] == time_pt]
        n_clusts = dat.shape[0]

        for i in range(n_clusts):
            cluster = dat['cluster'].iloc[i]
            ax.scatter(dat['lons'].iloc[i], dat['lats'].iloc[i], transform=ccrs.PlateCarree(), s=1, color=color_mapping[cluster], label=str(cluster), zorder=30)
            ax.annotate(text=str(cluster), xy=(dat['mean_lon'].iloc[i], dat['mean_lat'].iloc[i]), transform=ccrs.PlateCarree(), zorder=35)
        
        ax.legend()

    if (time_pt == practice_catalog.time).any():
        # grab time slice from Jonathan's raw AR catalogs
        catalog_slice = practice_catalog.sel(time=time_pt)
        ax.contour(catalog_slice.lon, catalog_slice.lat, catalog_slice, transform=ccrs.PlateCarree(), zorder=31, linestyles='dotted', colors='black', linewidths=1)
    
    ax.set_extent([-180,180,-90,-39], ccrs.PlateCarree())
    #land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m',edgecolor='none',facecolor='white') # 10m, 50m, 110m
    #ax.add_feature(land_50m,linewidth=3)
    ice_shelf_poly = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_polys', '50m',edgecolor='none',facecolor='lightcyan') # 10m, 50m, 110m
    ax.add_feature(ice_shelf_poly,linewidth=3)
    ice_shelf_line = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_lines', '50m',edgecolor='black',facecolor='none') # 10m, 50m, 110m
    ax.add_feature(ice_shelf_line,linewidth=1,zorder=13)
    ax.coastlines(resolution='110m',linewidth=1,zorder=32)

    
    # Map extent 
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.gridlines(alpha=0.5, zorder=33)
    
    time_ts = pd.Timestamp(time_pt)
    ax.set_title(f'{time_ts.month}/{time_ts.day}/{time_ts.year}, {time_ts.hour}:00')

In [ ]:
fig, ax = plt.subplots(figsize=(5,5), subplot_kw=dict(projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.)))
plt_time(0)
fig.suptitle('AR Identifier Test Movie', fontsize=16)

def update_img(i):
    ax.clear()
    plt_time(i)

ani = animation.FuncAnimation(fig, update_img, frames=len(times), interval = 200)

In [ ]:
def save_progress(i, n):
    if i%100 == 0: {print(f'Saving frame {i}/{n}')}

ani.save('identifier_test.mp4', progress_callback=save_progress)

In [3]:
Video('identifier_test.mp4')

### Test Movie

Pick the year 2022 (because that has the big AR associated with the heat wave!) Make another movie down here showcasing the identified ARs to see how good of a job it does on a dataset we didn't use to tune the algorithm. (**TBD**)

### Converting Back to One-Hot Encoded Format

In [5]:
curwd = os.getcwd()
identified_catalog = pd.read_pickle(curwd + '/cluster_infos_1980.pkl')

In [6]:
times = identified_catalog.time.unique()

In [7]:
da_lst = [None]*times.shape[0]
for i in range(times.shape[0]):
    single_df = identified_catalog[identified_catalog.time == times[i]]
    n_storms = single_df.shape[0]
    storm_df = [None]*n_storms

    for j in range(single_df.shape[0]):
        lats = pd.Series(single_df[['cluster', 'lats', 'lons']].lats.iloc[j])
        lons = pd.Series(single_df[['cluster', 'lats', 'lons']].lons.iloc[j])

        points = pd.DataFrame({'lat':lats, 'lon':lons})
        points['cluster'] = single_df['cluster'].iloc[j]
        storm_df[j] = points

    time_df = pd.concat(storm_df, axis=0)
    raster_day = time_df.set_index(['lat', 'lon']).to_xarray()
    da_lst[i] = raster_day

year_da = xr.concat(da_lst, dim='time')
year_da = year_da.assign_coords(time = times)
times = np.array(pd.date_range('1980-01-02T03:00:00.000000000', '1980-12-31T03:00:00.000000000', freq='3h'))
year_da = year_da.reindex(lat=practice_catalog.lat, lon=practice_catalog.lon, time=times)

year_da = year_da.fillna(0)
year_da = year_da.chunk('auto')